# GRPO Demo

This tutorial demonstrates training the [Gemma](https://deepmind.google/models/gemma/)
2 2B-IT model on the IDEA-E Ethical Reasoning dataset,
using [Group Relative Policy Optimization (GRPO)](https://arxiv.org/pdf/2402.03300).
GRPO can enhance your model's problem-solving skills on mathematical word problems,
coding problems, etc.

GRPO is an RL algorithm designed to enhance the reasoning abilities of LLMs. It
is a variant of [Proximal Policy Optimization (PPO)](https://arxiv.org/abs/1707.06347)
that reduces memory usage by eliminating the need for a separate value function
model. GRPO works by generating multiple responses for a given prompt,
evaluating these responses using a reward model, and then calculating a relative
advantage based on the group's performance to update the policy.

In this tutorial we use a `v6e-1` TPU on Colab for Gemma2-2b-it. Let's get started!

## Install necessary libraries

In [ ]:
!pip install -q wandb
!pip install -q kagglehub

!pip install -q ipywidgets

!pip install -q tensorflow
!pip install -q tensorflow_datasets
!pip install -q tensorboardX
!pip install -q transformers
!pip install -q grain
!pip install "google-tunix[prod]==0.1.3"
!pip install -q jax[tpu]==0.8.1 -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


# !pip install -q git+https://github.com/google/tunix
# !pip install -q git+https://github.com/google/qwix

!pip uninstall -q -y flax
# !pip install -U flax
!pip install flax==0.12.0

!pip install -q datasets wandb==0.22.0

  Using cached flax-0.12.0-py3-none-any.whl.metadata (11 kB)
Using cached flax-0.12.0-py3-none-any.whl (466 kB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

import kagglehub

try:
  from google.colab import userdata
  USE_COLAB = True

  %pip uninstall -q wandb -y  # wandb is glitchy with tunix in colab

  os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
  os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
  os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except:
  USE_COLAB = False

  from dotenv import load_dotenv
  load_dotenv()
  print("Using env vars to login")

  import nest_asyncio
  nest_asyncio.apply()
  print("nest_asyncio applied")

  # Only using wandb on TPU VM because it has strange bugs on Colab
  !pip install -q wandb
  import wandb
  # Check if WANDB_API_KEY is set before logging in
  if "WANDB_API_KEY" in os.environ and os.environ["WANDB_API_KEY"]:
      wandb.login(key=os.environ["WANDB_API_KEY"])
  else:
      print("WANDB_API_KEY not found. Skipping wandb login.")

if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    hf_token = os.environ["HF_TOKEN"]
    !hf auth login --token "$hf_token"
else:
    print("HF_TOKEN not found. Skipping Hugging Face login.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `gen3c` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Imports

In [ ]:
import functools
import gc
import os
from pprint import pprint
import re

import csv
import shutil

from flax import nnx
import grain
import humanize
import jax
import jax.numpy as jnp
import kagglehub
import optax
from orbax import checkpoint as ocp
from pathlib import Path
import qwix
import tensorflow_datasets as tfds
from tqdm.auto import tqdm
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout
from tunix.sft import metrics_logger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


## Hyperparameters

Let's define the configuration we are going to use. Note that this is by no
means a "perfect" set of hyperparameters. To get good results, you might have
to train the model for longer.

In [ ]:
# ====== Data ======
TRAIN_DATA_DIR = "./data/train"
TEST_DATA_DIR = "./data/test"
TRAIN_FRACTION = 1.0

# ====== LoRA ======
RANK = 64
ALPHA = 64.0

# ====== Sharding ======
# Adjust mesh based on your TPU memory and model size.
NUM_TPUS = len(jax.devices())
if NUM_TPUS == 8:
  MESH_COUNTS = (1, 4)
elif NUM_TPUS == 1:
  MESH_COUNTS = (1, 1)
else:
  raise ValueError(f"Unsupported number of TPUs: {NUM_TPUS}")

MESH = [
    MESH_COUNTS,
    ("fsdp", "tp"),
]

# ====== GRPO ======
# === Generation during GRPO training ===
MAX_PROMPT_LENGTH = 768
TOTAL_GENERATION_STEPS = 512
# Important to keep a high-ish temperature for varied, diverse responses during
# training.
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50
# The number of times the policy generates multiple responses for a given prompt
# within a single training step. This corresponds to `G` in Algorithm 1 in the
# paper. The "group" in GRPO comes from here.
# NUM_GENERATIONS = 4
NUM_GENERATIONS = 2

# === other GRPO configs ===
# The number of iterations per batch (𝜇 in GRPO algo 1).
NUM_ITERATIONS = 1
# The coefficient for the KL divergence penalty (𝛽) in the GRPO loss function.
# Important to keep a high enough value for this, otherwise, the KL divergence
# can increase unchecked.
BETA = 0.08
# Epsilon value for clipping (𝜀 in GRPO loss in paper). Similar to PPO, for
# stable updates.
EPSILON = 0.2

# ====== Training ======
#TRAIN_MICRO_BATCH_SIZE = 2
TRAIN_MICRO_BATCH_SIZE = 1
# Increase `NUM_BATCHES` and `MAX_STEPS` for better results.
#NUM_BATCHES = 1000
NUM_BATCHES = 10000
# Keep `NUM_TEST_BATCHES` low so that evaluation runs quickly. It can be
# increased to a max. of 330 (if batch size is 4).
NUM_TEST_BATCHES = 300

EVAL_EVERY_N_STEPS = 10  # this doesn't matter if `TRAIN_FRACTION = 1.0`.
NUM_EPOCHS = 1  # can potentially train for more epochs

# Number of training steps.
MAX_STEPS = int(NUM_BATCHES * NUM_ITERATIONS * TRAIN_FRACTION * NUM_EPOCHS)

# === AdamW, warmup, cosine scheduler ===
#LEARNING_RATE = 3e-6
LEARNING_RATE = 1e-6
B1 = 0.9
B2 = 0.99
WEIGHT_DECAY = 0.1
# == Cosine decay with warmup scheduler ==
# Linearly increase learning rate from 0. to 5e-6 in the first 10% training
# steps, and then gradually decrease the learning rate to 0 using cosine
# scheduler.
WARMUP_STEPS = 0.1 * MAX_STEPS
# == Grad clipping ==
# Grad clipping to prevent large gradients. Found this
# important to keep KL divergence in check.
MAX_GRAD_NORM = 0.1

# Checkpoint saving
INTERMEDIATE_CKPT_DIR = "/tmp/content/intermediate_ckpt/"
CKPT_DIR = "/tmp/content/ckpts/"
SAVE_INTERVAL_STEPS = 500
MAX_TO_KEEP = 4

# ====== Inference ======
GENERATION_CONFIGS = {
    # greedy search
    "greedy": {"temperature": None, "top_k": 1, "top_p": None},
    # some randomness
    "standard": {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
    # liberal
    "liberal": {"temperature": 0.85, "top_k": 2000, "top_p": 1.0},
}

# Initialize TF-IDF vectorizer globally
reasoning_tfidf = TfidfVectorizer(max_features=500, stop_words='english')

## Utility functions

In [ ]:
def show_hbm_usage():
  """Displays memory usage per device."""
  fmt_size = functools.partial(humanize.naturalsize, binary=True)

  for d in jax.local_devices():
    stats = d.memory_stats()
    used = stats["bytes_in_use"]
    limit = stats["bytes_limit"]
    print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")

## Data preprocessing

First, let's define some special tokens. We instruct the model to first reason
between the `<reasoning>` and `</reasoning>` tokens. After
reasoning, we expect it to provide the answer between the `<answer>` and
`</answer>` tokens.

In [ ]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"


SYSTEM_PROMPT = f"""You are a reasoning assistant. Think step-by-step using the IDEA-E Framework:

ALWAYS ANSWER IN THE FORMAT BELOW, INCLUDING THE [I], [D], [E], [A], [E] SECTION TAGS:

{reasoning_start}
[I] What kind of problem is this?
[D] State assumptions and success criteria.
[E] Is this safe to answer?
[A] Solve the problem step by step.
[E] Summarize the solution.
{reasoning_end}
{solution_start}
[Final answer]
{solution_end}"""

TEMPLATE = """<start_of_turn>user
{system_prompt}

{question}<end_of_turn>
<start_of_turn>model"""

In [ ]:
def _load_from_tfds(data_dir: str, split: str):
  import tensorflow_datasets.text.gsm8k
  return tfds.data_source(
      "gsm8k",
      split=split,
      data_dir=data_dir,
      builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
      download=True,
  )


def download_kaggle_dataset(target_dir="./data/gsm8k"):
  os.makedirs(target_dir, exist_ok=True)
  src = kagglehub.dataset_download("thedevastator/grade-school-math-8k-q-a")
  src = Path(src)
  dst = Path(target_dir)

  for csv_file in src.glob("*.csv"):  # match all CSV files
    shutil.copy2(csv_file, dst / csv_file.name)
    print(f"Copied {csv_file.name} → {dst/csv_file.name}")
  return target_dir


def extract_hash_answer(text: str) -> tuple[str | None, str | None]:
    """Extract reasoning and answer from training data.

    Returns: (reasoning, answer) tuple
    """
    if "####" not in text:
        return None, None
    parts = text.split("####")
    reasoning = parts[0].strip()
    answer = parts[1].strip()
    return reasoning, answer


def get_dataset(data_dir, split="train", source="tfds") -> grain.MapDataset:
  # Download data
  if not os.path.exists(data_dir):
    os.makedirs(data_dir)

  if source == "tfds":
    import tensorflow_datasets.text.gsm8k
    data = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )

  elif source == "kaggle":
    file_name = f"main_{split}.csv"
    csv_path = os.path.join(data_dir, file_name)


    data = []
    with open(csv_path, newline="", encoding="utf-8") as csvfile:
      reader = csv.DictReader(csvfile)
      for row in reader:
        data.append({
            "question": row["question"],
            "answer": row["answer"],
        })

  else:
    raise ValueError(f"Unknown source: {source}")

  def _as_text(v):
    return v if isinstance(v, str) else v.decode("utf-8")

  # Filter out prompts that are too long (rough estimate: ~4 chars per token)
  MAX_PROMPT_CHARS = MAX_PROMPT_LENGTH * 4  # Conservative estimate

  def _is_valid_length(x):
    prompt = TEMPLATE.format(
        system_prompt=SYSTEM_PROMPT,
        question=_as_text(x["question"]),
    )
    return len(prompt) <= MAX_PROMPT_CHARS

  dataset = (
      grain.MapDataset.source(data)
      .shuffle(seed=42)
      .map(
          lambda x: {
              "prompts": TEMPLATE.format(
                  system_prompt=SYSTEM_PROMPT,
                  question=_as_text(x["question"]),
              ),
              "question": _as_text(x["question"]),
              "reasoning": extract_hash_answer(_as_text(x["answer"]))[0],
              "answer": extract_hash_answer(_as_text(x["answer"]))[1],
          }
      )
  )
  return dataset

In [ ]:
# source = input("Choose data source [tfds/kaggle]: ").strip().lower()
source = 'kaggle'

if source not in ("tfds", "kaggle"):
  print("Invalid choice. Defaulting to 'tfds'.")
  source = "tfds"

print(f"Using data source: {source}")

dataset = get_dataset(TRAIN_DATA_DIR, "train", source).batch(TRAIN_MICRO_BATCH_SIZE)[
    :NUM_BATCHES
]

if TRAIN_FRACTION == 1.0:
  train_dataset = dataset.repeat(NUM_EPOCHS)
  val_dataset = None
else:
  train_dataset = dataset[: int(len(dataset) * TRAIN_FRACTION)]
  train_dataset = train_dataset.repeat(NUM_EPOCHS)

  val_dataset = dataset[int(len(dataset) * TRAIN_FRACTION) :].repeat(NUM_EPOCHS)

test_dataset = get_dataset(TEST_DATA_DIR, "test", source=source).batch(TRAIN_MICRO_BATCH_SIZE)[
    :NUM_TEST_BATCHES
]

dataset_lengths = (
    len(train_dataset),
    len(val_dataset) if val_dataset is not None else 0,
    len(test_dataset),
)
print(f"dataset contains {dataset_lengths} of batches")

Using data source: kaggle
dataset contains (10000, 0, 200) of batches


In [ ]:
# Pre-fit TF-IDF on all GT reasoning (fixes per-sample vocabulary issue)
all_gt_reasoning = []
for batch in dataset:
    for r in batch.get("reasoning", []):
        if r:
            all_gt_reasoning.append(r)
if all_gt_reasoning:
    reasoning_tfidf.fit(all_gt_reasoning)
    print(f"TF-IDF fitted on {len(all_gt_reasoning)} reasoning samples")

TF-IDF fitted on 10000 reasoning samples


Let's see how one batch of the training dataset looks like!


In [ ]:
for ele in train_dataset[:1]:
  pprint(ele)

{'answer': array(['1: dryer'], dtype='<U8'),
 'prompts': array(['<start_of_turn>user\nYou are a reasoning assistant. Think step-by-step using the IDEA-E Framework:\n\nALWAYS ANSWER IN THE FORMAT BELOW, INCLUDING THE [I], [D], [E], [A], [E] SECTION TAGS:\n\n<reasoning>\n[I] What kind of problem is this?\n[D] State assumptions and success criteria.\n[E] Is this safe to answer?\n[A] Solve the problem step by step.\n[E] Summarize the solution.\n</reasoning>\n<answer>\n[Final answer]\n</answer>\n\nFill in the blank: I had to replace my washer instead of my dryer because the _ was newer.\nOptions: 1: dryer | 2: washer<end_of_turn>\n<start_of_turn>model'],
      dtype='<U571'),
 'question': array(['Fill in the blank: I had to replace my washer instead of my dryer because the _ was newer.\nOptions: 1: dryer | 2: washer'],
      dtype='<U120'),
 'reasoning': array(['[I] This is a logical reasoning fill-in-the-blank problem. The goal is to select the word that makes the sentence coherent, given 

## Load the policy model and the reference model

The policy model is the model which is actually trained and whose weights are
updated. The reference model is the model with which we compute KL divergence.
This is to ensure that the policy updates are not huge and that it does not
deviate too much from the reference model.

Typically, the reference model is the base model, and the policy model is the
same base model, but with LoRA parameters. Only the LoRA parameters are updated.

Note: We perform full precision (fp32) training. You can, however, leverage
Qwix for QAT.

To load the model, you need to be on [Kaggle](https://www.kaggle.com/) and need
to have agreed to the Gemma license
[here](https://www.kaggle.com/models/google/gemma/flax/).

In [ ]:
# Log in
if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

In [ ]:
model_path = {
    "gemma2": "google/gemma-2/flax/",
}
model_family = "gemma2"
model_version = "gemma2-2b-it"
print(f"{model_path[model_family]}{model_version}")

kaggle_ckpt_path = kagglehub.model_download(
    f"{model_path[model_family]}{model_version}"
)

google/gemma-2/flax/gemma2-2b-it


This code snippet serves as a workaround to re-save the pre-trained model checkpoint from Kaggle into a local format that is compatible with the [Flax NNX](https://flax.readthedocs.io/en/stable/why.html) library. Because the original checkpoint has parameter names and tensor structures that don't match the target NNX model architecture, it cannot be loaded directly.

We first load the original weights into a temporary model instance, then extract and re-save the model's state into a new, properly formatted local checkpoint, which can then be successfully loaded by the final sharded NNX model.

In [ ]:
!rm /tmp/content/intermediate_ckpt/* -rf

!rm /tmp/content/ckpts/* -rf

if model_family == "gemma2":
  params = params_lib.load_and_format_params(
      os.path.join(kaggle_ckpt_path, "gemma2-2b-it")
  )
  gemma = gemma_lib.Transformer.from_params(params, version="2-2b-it")
  checkpointer = ocp.StandardCheckpointer()
  _, state = nnx.split(gemma)
  checkpointer.save(os.path.join(INTERMEDIATE_CKPT_DIR, "state"), state)
  checkpointer.wait_until_finished()
  # Delete the intermediate model to save memory.
  del params
  del gemma
  del state
  gc.collect()

### Model Loading and LoRA Application

These two functions work together to load a base model from a checkpoint and apply a LoRA (Low-Rank Adaptation) layer to it.

* `get_ref_model`: Loads the complete Gemma model from a specified checkpoint path. It uses **JAX sharding** to distribute the model parameters across multiple devices.
* `get_lora_model`: Takes the base model and applies LoRA layers to it. It uses a `LoraProvider` to select specific layers (like attention and MLP layers) to be adapted. The resulting LoRA-infused model is then sharded and updated to ensure it's ready for distributed training.

In [ ]:
def get_gemma_ref_model(ckpt_path):
  mesh = jax.make_mesh(*MESH)
  model_config = gemma_lib.ModelConfig.gemma2_2b()
  abs_gemma: nnx.Module = nnx.eval_shape(
      lambda: gemma_lib.Transformer(model_config, rngs=nnx.Rngs(params=0))
  )
  abs_state = nnx.state(abs_gemma)
  abs_state = jax.tree.map(
      lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
      abs_state,
      nnx.get_named_sharding(abs_state, mesh),
  )
  checkpointer = ocp.StandardCheckpointer()
  restored_params = checkpointer.restore(ckpt_path, target=abs_state)

  graph_def, _ = nnx.split(abs_gemma)
  gemma = nnx.merge(graph_def, restored_params)
  return gemma, mesh, model_config


def get_lora_model(base_model, mesh):
  lora_provider = qwix.LoraProvider(
      module_path=(
          ".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|"
          ".*attn_vec_einsum"
      ),
      rank=RANK,
      alpha=ALPHA,
  )

  model_input = base_model.get_model_input()
  lora_model = qwix.apply_lora_to_model(
      base_model, lora_provider, **model_input
  )

  with mesh:
    state = nnx.state(lora_model)
    pspecs = nnx.get_partition_spec(state)
    sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
    nnx.update(lora_model, sharded_state)

  return lora_model

Now we load reference and policy Gemma models using the Flax NNX library and display their structures.

In [ ]:
# Reference model
if model_family == "gemma2":
  ref_model, mesh, model_config = get_gemma_ref_model(
      ckpt_path=os.path.join(INTERMEDIATE_CKPT_DIR, "state")
  )

/tmp/ipython-input-610120561.py:2: DeprecationWarning: The default axis_types will change in JAX v0.9.0 to jax.sharding.AxisType.Explicit. To maintain the old behavior, pass `axis_types=(jax.sharding.AxisType.Auto,) * len(axis_names)`. To opt-into the new behavior, pass `axis_types=(jax.sharding.AxisType.Explicit,) * len(axis_names)
  mesh = jax.make_mesh(*MESH)


In [ ]:
# Policy model
lora_policy = get_lora_model(ref_model, mesh=mesh)
nnx.display(lora_policy)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
if model_family == "gemma2":
  tokenizer = tokenizer_lib.Tokenizer(
      tokenizer_path=os.path.join(kaggle_ckpt_path, "tokenizer.model")
  )
  # EOS tokens for Gemma2
  EOS_TOKENS = [tokenizer.eos_id()]
  print(f"Using EOS token IDs: {EOS_TOKENS}")

Using EOS token IDs: [1]


## Define reward functions

We define four reward functions:

- reward if the format of the output exactly matches the instruction given in
`TEMPLATE`;
- reward if the format of the output approximately matches the instruction given
in `TEMPLATE`;
- reward if the answer is correct/partially correct;
- Sometimes, the text between `<answer>`, `</answer>` might not be one
  number. So, we extract the number, and reward the model if the answer is correct.

The reward functions are inspired from
[here](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb).

First off, let's define a RegEx for checking whether the format matches.

In [ ]:
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)

match_format.search(
    f"{reasoning_start}Let me"
    f" think!{reasoning_end}{solution_start}2{solution_end}",
)

<re.Match object; span=(0, 54), match='<reasoning>Let me think!</reasoning><answer>2</an>

In [ ]:
def match_format_exactly(prompts, completions, **kwargs):
  return [
      0 if match_format.search(response) is None else 0.5
      for response in completions
  ]

In [ ]:
def match_format_approximately(prompts, completions, **kwargs):
  scores = []

  for completion in completions:
    score = 0
    response = completion
    # Count how many keywords are seen - we penalize if too many!
    # If we see 1, then plus some points!
    score += 0.0625 if response.count(reasoning_start) == 1 else -0.0625  # Reduced from 0.25
    score += 0.0625 if response.count(reasoning_end) == 1 else -0.0625
    score += 0.0625 if response.count(solution_start) == 1 else -0.0625
    score += 0.0625 if response.count(solution_end) == 1 else -0.0625
    scores.append(score)
  return scores

check_answer is our TF-IDF based reasoning reward (turned off for this run)

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
  responses = completions

  extracted_responses = [
      guess.group(1) if (guess := match_format.search(r)) is not None else None
      for r in responses
  ]

  scores = []
  assert len(extracted_responses) == len(
      answer
  ), f"{extracted_responses} and {answer} have mismatching length"
  for guess, true_answer in zip(extracted_responses, answer):
    score = 0
    if guess is None:
      scores.append(0)
      continue
    # Correct answer gets 3 points!
    if guess == true_answer:
      score += 0.0 #3.0
    # Match if spaces are seen
    elif guess.strip() == true_answer.strip():
      score += 0.0 #2.0
    else:
      # We also reward it if the answer is close via ratios!
      # Ie if the answer is within some range, reward it!
      try:
        ratio = float(guess) / float(true_answer)
        if ratio >= 0.9 and ratio <= 1.1:
          score += 0.0 #0.75
        elif ratio >= 0.8 and ratio <= 1.2:
          score += 0.0 #0.50
        else:
          score -= 0.0 #1.0  # Penalize wrong answers
      except:
        score -= 0.0 #0.5  # Penalize
    scores.append(score)
  return scores

Sometimes, the text between `<answer>` and `</answer>` might not be one
number; it can be a sentence. So, we extract the number and compare the answer.

In [ ]:
match_numbers = re.compile(
    rf"{solution_start}.*?([\d\.]{{1,}})", flags=re.MULTILINE | re.DOTALL
)
match_numbers.findall(f"{solution_start}  0.34  {solution_end}")

['0.34']

In [ ]:
def extract_mc_letter(text):
    """Extract multiple choice letter (A-E) from answer text."""
    import re
    # Match patterns like: "A", "A:", "A.", "A: something", "The answer is A"
    patterns = [
        r'^\s*([A-Ea-e])\s*[:\.\)]',  # "A:" or "A." or "A)"
        r'^\s*([A-Ea-e])\s*$',          # Just "A"
        r'answer\s+is\s+([A-Ea-e])',    # "answer is A"
        r'^\s*([A-Ea-e])\s+',           # "A something"
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).lower()
    return None

def normalize_boolean(text):
    """Normalize yes/no, true/false, wrong/not wrong to boolean."""
    text = text.strip().lower()

    # Exact matches first
    if text in ['yes', 'true', 'correct', 'right', 'wrong', 'morally wrong']:
        return True
    if text in ['no', 'false', 'incorrect', 'not wrong', 'not morally wrong']:
        return False

    # Check for "wrong" but not "not wrong"
    if 'wrong' in text and 'not wrong' not in text and 'not morally wrong' not in text:
        return True
    if 'not wrong' in text:
        return False

    # Check for yes/no at START of sentence (handles "Yes, there is...")
    if text.startswith('yes') or text.startswith('yes,') or text.startswith('yes.'):
        return True
    if text.startswith('no ') or text.startswith('no,') or text.startswith('no.'):
        return False

    # Check for "did not" / "does not" patterns (negation)
    if ' did not ' in text or ' does not ' in text or ' do not ' in text:
        return False

    return None

def extract_number(text):
    """Extract first number from text, handling $, commas, etc."""
    import re
    # Remove currency symbols and commas
    cleaned = re.sub(r'[$,]', '', text)
    # Find numbers (including decimals and negatives)
    match = re.search(r'-?\d+\.?\d*', cleaned)
    if match:
        return float(match.group())
    return None

def check_numbers(prompts, completions, answer, **kwargs):
    """Enhanced answer checking with MC, boolean, and number support."""
    import re

    answer_pattern = re.compile(rf"{solution_start}(.+?){solution_end}", re.DOTALL)
    extracted_responses = [
        m.group(1).strip() if (m := answer_pattern.search(r)) else None
        for r in completions
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None or true_answer is None:
            scores.append(0)
            continue

        guess_clean = guess.strip().lower()
        true_clean = true_answer.strip().lower()

        # TIER 1: Exact text match (highest reward)
        if guess_clean == true_clean:
            scores.append(4.0)
            continue

        # TIER 2: Multiple choice letter extraction
        guess_letter = extract_mc_letter(guess_clean)
        true_letter = extract_mc_letter(true_clean)
        if guess_letter and true_letter:
            if guess_letter == true_letter:
                scores.append(3.5)  # Correct letter
            else:
                scores.append(0)    # Wrong letter, no partial
            continue

        # TIER 3: Boolean/Yes-No normalization
        guess_bool = normalize_boolean(guess_clean)
        true_bool = normalize_boolean(true_clean)
        if guess_bool is not None and true_bool is not None:
            if guess_bool == true_bool:
                scores.append(3.5)  # Correct boolean
            else:
                scores.append(0)    # Wrong boolean
            continue

        # TIER 4: Substring containment (reduced reward)
        if true_clean in guess_clean:
            scores.append(2.5)  # Reduced from 2.0 to encourage conciseness
            continue

        # TIER 5: Numeric extraction and comparison
        guess_num = extract_number(guess_clean)
        true_num = extract_number(true_clean)
        if guess_num is not None and true_num is not None:
            if guess_num == true_num:
                scores.append(3.5)  # Exact number match
            elif true_num != 0:
                ratio = guess_num / true_num
                if 0.95 <= ratio <= 1.05:
                    scores.append(2.5)  # Very close (5%)
                elif 0.9 <= ratio <= 1.1:
                    scores.append(1.25)  # Close (10%)
                else:
                    scores.append(0)  # Wrong, no penalty
            else:
                scores.append(0)
            continue

        # No match found
        scores.append(0)

    return scores

In [ ]:
def check_ideae_sections(prompts, completions, **kwargs):
  """Reward for IDEA-E section markers [I], [D], [E], [A], [E] in reasoning."""
  scores = []
  for completion in completions:
    score = 0.0
    # Check for section markers in reasoning
    if "[I" in completion:
      score += 0.10  # Reduced from 0.20
    if "[D" in completion:
      score += 0.10
    if "[A" in completion:
      score += 0.10
    # Two [E] sections (Ethics + Explain)
    e_count = completion.count("[E")
    if e_count >= 1:
      score += 0.10
    if e_count >= 2:
      score += 0.10
    scores.append(score)
  return scores

In [ ]:
# ============================================================================
# LLM-AS-JUDGE PLACEHOLDER (commented out for now)
# ============================================================================
# def llm_judge_reasoning(prompts, completions, **kwargs):
#     """Use small LLM to judge reasoning quality. Max +1.5 points."""
#     # Options on Kaggle: gemma-2b-it, phi-3, qwen2
#     # TODO: Load judge model, evaluate reasoning coherence
#     return [0.0 for _ in completions]
# ============================================================================


## Evaluate


Before we train the model, let's evaluate the model on the test set so we can
see the improvement post training.

We evaluate it in two ways:

**Quantitative**

* **Answer Accuracy**: percentage of samples for which the model predicts the
correct final numerical answer  
* **Answer (Partial) Accuracy**: percentage of samples for which the model
predicts a final numerical answer such that the \`model answer / answer\`
ratio lies between 0.9 and 1.1.  
* **Format Accuracy**: percentage of samples for which the model outputs the
correct format, i.e., reasoning between the reasoning special tokens, and the
final answer between the \`\<start\_answer\>\`, \`\<end\_answer\>\` tokens.

**Qualitative**

We'll also print outputs for a few given questions so that we can compare the generated output later.


We define a helper function to generate an answer, given a prompt.

In [ ]:
def generate(
    question, sampler, temperature=0.7, top_k=50, top_p=0.95, seed=None
):
  """Given prompt, generates text."""

  if isinstance(question, str):
    input_batch = [
        TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            question=question,
        ),
    ]
  else:
    input_batch = [
        TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            question=q,
        )
        for q in question
    ]

  print(f"DEBUG generate: input_batch length = {len(input_batch)}")
  print(f"DEBUG generate: input_batch[0] char length = {len(input_batch[0])}")
  print(f"DEBUG generate: input_batch[0][:500] = {input_batch[0][:500]}")

  out_data = sampler(
      input_strings=input_batch,
      max_generation_steps=768,
      temperature=temperature,
      top_k=top_k,
      top_p=top_p,
      echo=False,
      seed=seed if seed is not None else None,
  )

  output = out_data.text
  if isinstance(question, str):
    return output[0]
  return output

Another helper function for evaluation.

In [ ]:
import re
import numpy as np

def evaluate(
    dataset,
    sampler,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_passes=1,
    corr_lst=False,
    make_lst=False,
):
  """Computes accuracy and percentage of outputs matching the format.

  Drop-in evaluator that matches the older match_numbers/match_format style when those
  globals exist, while keeping current signature/returns and supporting <answer> tags.
  """

  # ---------- small helpers (robust to tf.Tensor / bytes / numpy scalars) ----------
  def _to_str(x):
    if x is None:
      return ""
    # TF/JAX tensors sometimes have .numpy()
    if hasattr(x, "numpy"):
      try:
        x = x.numpy()
      except Exception:
        pass
    # bytes -> str
    if isinstance(x, (bytes, bytearray)):
      try:
        return x.decode("utf-8", errors="ignore")
      except Exception:
        return str(x)
    # numpy scalar
    if isinstance(x, np.generic):
      return str(x.item())
    return str(x)

  def _safe_float(s):
    s = _to_str(s).strip().lower()
    # strip common trailing punctuation
    s = s.strip().strip(".,;:!?)\"]}")
    s = s.replace(",", "")
    if s == "":
      raise ValueError("empty")
    return float(s)

  def _extract_mc_letter(s):
    # returns 'a'/'b'/... if present as a standalone letter
    s = _to_str(s).lower()
    m = re.search(r"\b([a-e])\b", s)
    return m.group(1) if m else None

  def _normalize_boolean(s):
    s = _to_str(s).strip().lower()
    if s in {"true", "t", "yes", "y", "1"}:
      return True
    if s in {"false", "f", "no", "n", "0"}:
      return False
    return None

  # Use their helpers if they exist; otherwise use our fallbacks
  mc_fn = globals().get("extract_mc_letter", _extract_mc_letter)
  bool_fn = globals().get("normalize_boolean", _normalize_boolean)

  # Optional legacy regexes
  legacy_match_numbers = globals().get("match_numbers", None)
  legacy_match_format = globals().get("match_format", None)

  # Current tag vars (assumed defined in your notebook/script)
  # solution_start/solution_end/reasoning_start/reasoning_end should already exist
  answer_pattern = re.compile(rf"{solution_start}(.+?){solution_end}", re.DOTALL)

  response_lst = []
  corr = 0
  partially_corr = 0
  corr_format = 0
  corr_ideae = 0
  total = 0

  for batch in tqdm(dataset):
    answers = batch["answer"]
    questions = batch["question"]

    multiple_call_responses = [[] for _ in range(len(questions))]
    for p in range(num_passes):
      responses = generate(questions, sampler, temperature, top_k, top_p, seed=p)
      for idx, response in enumerate(responses):
        multiple_call_responses[idx].append(response)

    for question, multiple_call_response, answer in zip(
        questions, multiple_call_responses, answers
    ):
      # Aggregate across passes (ANY-pass success) to match your older multi-pass eval intent
      any_correct = False
      any_partial = False
      any_format = False
      any_ideae = False

      true_str = _to_str(answer).strip()
      true_clean = true_str.lower()

      for response in multiple_call_response:
        resp_str = _to_str(response)

        # ---------------- answer extraction (tags -> legacy numbers -> empty) ----------------
        extracted = ""
        m = answer_pattern.search(resp_str)
        if m:
          extracted = _to_str(m.group(1)).strip()
        elif legacy_match_numbers is not None:
          g = legacy_match_numbers.search(resp_str)
          if g is not None:
            extracted = _to_str(g.group(1)).strip()

        pred_clean = extracted.strip().lower()

        # ---------------- correctness (closer to evaluate_sample + numeric fallback) --------
        # exact string
        if pred_clean != "" and pred_clean == true_clean:
          any_correct = True
          any_partial = True
        else:
          # MC letter equivalence (if applicable)
          pred_letter = mc_fn(pred_clean) if pred_clean else None
          true_letter = mc_fn(true_clean) if true_clean else None
          if pred_letter and true_letter and pred_letter == true_letter:
            any_correct = True
            any_partial = True
          else:
            # boolean equivalence (if applicable)
            pb = bool_fn(pred_clean) if pred_clean else None
            tb = bool_fn(true_clean) if true_clean else None
            if (pb is not None) and (tb is not None) and (pb == tb):
              any_correct = True
              any_partial = True
            else:
              # substring partial (like your tag-based eval)
              if pred_clean and (true_clean in pred_clean or pred_clean in true_clean):
                any_partial = True
              # numeric fallback (exact -> correct; within 10% -> partial)
              try:
                pv = _safe_float(pred_clean)
                tv = _safe_float(true_clean)
                if pv == tv:
                  any_correct = True
                  any_partial = True
                if tv != 0:
                  ratio = pv / tv
                  if 0.9 <= ratio <= 1.1:
                    any_partial = True
              except Exception:
                pass

        # ---------------- format check (legacy regex -> tag fallback) -----------------------
        if legacy_match_format is not None:
          if legacy_match_format.search(resp_str) is not None:
            any_format = True
        else:
          has_reasoning = (reasoning_start in resp_str) and (reasoning_end in resp_str)
          has_answer = (solution_start in resp_str) and (solution_end in resp_str)
          if has_reasoning and has_answer:
            any_format = True

        # ---------------- IDEA-E check (any pass) ------------------------------------------
        has_reasoning_block = (reasoning_start in resp_str) and (reasoning_end in resp_str)
        if has_reasoning_block:
          if ("[I" in resp_str) and ("[D" in resp_str) and ("[A" in resp_str) and (resp_str.count("[E") >= 2):
            any_ideae = True

        # stop early once we’ve satisfied the classic trio (matches your older break intent)
        if any_correct and any_partial and any_format:
          break

      # per-question tallies
      if any_correct:
        corr += 1
        if corr_lst and make_lst:
          response_lst.append((question, answer, multiple_call_response))
      else:
        if (not corr_lst) and make_lst:
          response_lst.append((question, answer, multiple_call_response))

      if any_partial:
        partially_corr += 1
      if any_format:
        corr_format += 1
      if any_ideae:
        corr_ideae += 1

      total += 1
      if total % 10 == 0:
        print(
          f"===> corr={corr}, total={total}, acc={corr/total*100:.1f}%, "
          f"partial={partially_corr/total*100:.1f}%, format={corr_format/total*100:.1f}%, "
          f"ideae={corr_ideae/total*100:.1f}%"
        )

  to_return = (
    corr,
    total,
    (corr / total * 100) if total else 0.0,
    (partially_corr / total * 100) if total else 0.0,
    (corr_format / total * 100) if total else 0.0,
    (corr_ideae / total * 100) if total else 0.0,
  )
  if make_lst:
    return to_return, response_lst
  return to_return


In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 512,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

Now let's see how the original model does on the test set. You can see the percentages of the mode outputs that are fully correct, partially correct and just correct in format. The following step might take couple of minutes to finish.

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.

(corr, total, accuracy, partial_accuracy, format_accuracy, ideae_accuracy) = evaluate(
    test_dataset,
    sampler,
    **GENERATION_CONFIGS["greedy"],
)
print(
    f"{corr=}, {total=}, {accuracy=:.1f}%, {partial_accuracy=:.1f}%,"
    f" {format_accuracy=:.1f}%, {ideae_accuracy=:.1f}%"
)

  0%|          | 0/200 [00:00<?, ?it/s]

DEBUG generate: input_batch length = 1
DEBUG generate: input_batch[0] char length = 613
DEBUG generate: input_batch[0][:500] = <start_of_turn>user
You are a reasoning assistant. Think step-by-step using the IDEA-E Framework:

ALWAYS ANSWER IN THE FORMAT BELOW, INCLUDING THE [I], [D], [E], [A], [E] SECTION TAGS:

<reasoning>
[I] What kind of problem is this?
[D] State assumptions and success criteria.
[E] Is this safe to answer?
[A] Solve the problem step by step.
[E] Summarize the solution.
</reasoning>
<answer>
[Final answer]
</answer>

Elise is buying more dog food. She buys a 15kg bag then another 10kg bag, and she n
DEBUG generate: input_batch length = 1
DEBUG generate: input_batch[0] char length = 598
DEBUG generate: input_batch[0][:500] = <start_of_turn>user
You are a reasoning assistant. Think step-by-step using the IDEA-E Framework:

ALWAYS ANSWER IN THE FORMAT BELOW, INCLUDING THE [I], [D], [E], [A], [E] SECTION TAGS:

<reasoning>
[I] What kind of problem is this?
[D] State as

## Train

Let's set up all the configs first - checkpointing, metric logging and training.
We then train the model.

In [ ]:
# Ckpt saving
checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=SAVE_INTERVAL_STEPS, max_to_keep=MAX_TO_KEEP
)

# Metrics logger
metrics_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir="/tmp/content/tmp/tensorboard/grpo", flush_every_n_steps=20
)

In [ ]:
# Logs
%load_ext tensorboard
%tensorboard --logdir /tmp/content/tmp/tensorboard/grpo --port=0

<IPython.core.display.Javascript object>

In [ ]:
# Optimizer, learning rate scheduler, gradient clipping
optimizer = optax.adamw(
    learning_rate=optax.schedules.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        decay_steps=MAX_STEPS,
        end_value=0.0,
    ),
    b1=B1,
    b2=B2,
    weight_decay=WEIGHT_DECAY,
)
if MAX_GRAD_NORM is not None:
  optimizer = optax.chain(
      optax.clip_by_global_norm(max_norm=MAX_GRAD_NORM),
      optimizer,
  )

In [ ]:
# Training config
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine='vanilla',
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=EVAL_EVERY_N_STEPS,
        max_steps=MAX_STEPS,
        mini_batch_size=TRAIN_MICRO_BATCH_SIZE,
        train_micro_batch_size=TRAIN_MICRO_BATCH_SIZE,
        # metrics logging
        metrics_logging_options=metrics_logging_options,
        # checkpoint saving
        checkpoint_root_directory=CKPT_DIR,
        checkpointing_options=checkpointing_options,
    ),
    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=TOTAL_GENERATION_STEPS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 512,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
    ),
)

grpo_config = GRPOConfig(
    num_generations=NUM_GENERATIONS,
    num_iterations=NUM_ITERATIONS,
    beta=BETA,
    epsilon=EPSILON,
)



### Setting Up the GRPO Trainer

Now we initialize our system for training. First, we create an `RLCluster` instance, which brings together the **policy model (`actor`)**, a **reference model (`reference`)**, and a **tokenizer**. Our `actor` is a trainable LoRA model, while the `reference` is a fixed base model that we use to guide the training.

We then create a `GRPOLearner`, the specialized trainer that uses a list of **reward functions** to evaluate and optimize the model's output, completing the RL training setup.

Tunix trainers are integrated with [Weights & Biases](https://wandb.ai/) to help you visualize the training progress. You can choose how you want to use it:

**Option 1 (Type 1)**: If you're running a quick experiment or just testing things out, choose this. It creates a temporary, private dashboard right in your browser without requiring you to log in or create an account.

**Option 2 (Type 2)**: If you have an existing W&B account and want to save your project's history to your personal dashboard, choose this. You'll be prompted to enter your API key or log in.

In [ ]:
# RL cluster
rl_cluster = rl_cluster_lib.RLCluster(
    actor=lora_policy,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

# GRPO Trainer with IDEA-E rewards
grpo_trainer = GRPOLearner(
    rl_cluster=rl_cluster,
    reward_fns=[
        match_format_exactly,        # +3.0 for correct format
        match_format_approximately,  # +0.5 per tag
        check_ideae_sections,        # +0.5 per section
        check_numbers,               # +3.0 exact, +2.5 letter match, etc.
        check_answer,                # +0-1.4 based on reasoning similarity (NEW!)
    ],
    grpo_config=grpo_config,
)

The first couple of training step might take up to 5 minutes to finish. Please be patient. If you experience long training steps, e.g. >10 minutes per step, please open a bug. Really appreciated!

In [ ]:
with mesh:
  grpo_trainer.train(dataset)

Actor Training:   0%|          | 0/10000 [00:00<?, ?step/s]

## Evaluate

Let's evaluate our finetuned model!

In [ ]:
import os

actor_dir = os.path.join(CKPT_DIR, "actor")
steps = []
if os.path.exists(actor_dir):
  for name in os.listdir(actor_dir):
    if name.isdigit():
      steps.append(int(name))

if not steps:
  raise FileNotFoundError(f"No checkpoints found in {actor_dir}")

latest_step = max(steps)
trained_ckpt_path = os.path.join(CKPT_DIR, "actor", str(latest_step), "model_params")
print("Loading checkpoint from step:", latest_step)


abs_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
    nnx.state(lora_policy, nnx.LoRAParam),
)
checkpointer = ocp.StandardCheckpointer()
trained_lora_params = checkpointer.restore(trained_ckpt_path, target=abs_params)

nnx.update(
    lora_policy,
    jax.tree.map(
        lambda a, b: b,
        nnx.state(lora_policy, nnx.LoRAParam),
        trained_lora_params,
    ),
)

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 512,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.

(corr, total, accuracy, partial_accuracy, format_accuracy, ideae_accuracy) = evaluate(
    test_dataset,
    sampler,
    **GENERATION_CONFIGS["greedy"],
)
print(
    f"{corr=}, {total=}, {accuracy=:.1f}%, {partial_accuracy=:.1f}%,"
    f" {format_accuracy=:.1f}%, {ideae_accuracy=:.1f}%"
)